# Assignment 4

**Rules**
* Put your code in the code cells and your written answers in the markdown cells below each part. Do not delete the task text.
* Run the whole notebook top to bottom (**Kernel > Restart & Run All**) before submitting; outputs must be visible.
* Use **Spark SQL (`spark.sql`)** for the SQL questions unless a part says Dask or DataFrame API.
* Do not hard-code results. Every number in your answers must come from a query.
* Q5 uses the documents you create in Q4(a). Run Q4 first.

## Setup (provided)

In [1]:
import os, json, glob, shutil
import pandas as pd

HADOOP_PATH = "C:\\hadoop"
os.environ["HADOOP_HOME"] = HADOOP_PATH

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("sleep-assignment")
         .master("local[*]").config("spark.sql.shuffle.partitions", "8").config("spark.hadoop.hadoop.home.dir", HADOOP_PATH).getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

CSV_PATH = "sleep_debt_screen_time.csv"      # change if your file is elsewhere
OUT = "assignment_out"                       # folder for files written by this notebook
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)

C:\Users\empty\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.1


In [2]:
df = (spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH))

# blue_light_filter_on is True/False in the file: convert it to 1/0 so it can be summed and averaged
df = df.withColumn("blue_light_filter_on",
                   F.when(F.lower(F.col("blue_light_filter_on").cast("string")) == "true", 1).otherwise(0))

df.createOrReplaceTempView("sleep")           # the SQL view used in Q1-Q3
print("rows:", df.count())
df.printSchema()

rows: 8500
root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- chronotype: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- bedtime_hour: double (nullable = true)
 |-- screen_time_before_bed_min: integer (nullable = true)
 |-- primary_app_category: string (nullable = true)
 |-- screen_brightness_pct: integer (nullable = true)
 |-- blue_light_filter_on: integer (nullable = false)
 |-- caffeine_intake_mg: integer (nullable = true)
 |-- caffeine_last_hour: double (nullable = true)
 |-- physical_activity_min: integer (nullable = true)
 |-- room_temperature_f: double (nullable = true)
 |-- ambient_noise_db: double (nullable = true)
 |-- sleep_onset_latency_min: double (nullable = true)
 |-- total_sleep_hours: double (nullable = true)
 |-- rem_sleep_pct: double (nullable = true)
 |-- deep_sleep_pct: double (nullable = true)
 |-- awakenings_count: integer (nullable = true)
 |-- sleep_efficiency_pct:

---
# Q1. SQL basics (20 points)
Use the view `sleep`.

### Q1(a) Dataset check (4 points)
Write **one** Spark SQL query that returns a single row with: `total_rows`, `distinct_users` (`COUNT(DISTINCT user_id)`), `min_sleep`, `avg_sleep`, `max_sleep` (of `total_sleep_hours`, rounded to 2 decimals).
Then, in a markdown cell, state whether `user_id` is a valid unique key.

In [3]:
spark.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT user_id) AS distinct_users,
        ROUND(MIN(total_sleep_hours), 2) AS min_sleep,
        ROUND(AVG(total_sleep_hours), 2) AS avg_sleep,
        ROUND(MAX(total_sleep_hours), 2) AS max_sleep
    FROM sleep
""").show()

+----------+--------------+---------+---------+---------+
|total_rows|distinct_users|min_sleep|avg_sleep|max_sleep|
+----------+--------------+---------+---------+---------+
|      8500|          8500|     3.91|     6.78|     9.69|
+----------+--------------+---------+---------+---------+



**Your answer:** user_id is a valid key, as total_rows has the same number of rows as distinct_users.

### Q1(b) Filtering (4 points)
Find users who used a screen for **more than 90 minutes** before bed, had the **blue-light filter off**, and had **sleep efficiency below 75%**.
Return `user_id, age, primary_app_category, screen_time_before_bed_min, sleep_efficiency_pct`, ordered by `sleep_efficiency_pct` ascending, **10 rows**.

In [4]:
spark.sql("""
    SELECT user_id, age, primary_app_category, screen_time_before_bed_min, sleep_efficiency_pct
    FROM sleep
    WHERE screen_time_before_bed_min > 90 
      AND blue_light_filter_on = 0 
      AND sleep_efficiency_pct < 75
    ORDER BY sleep_efficiency_pct ASC
    LIMIT 10
""").show()

+-------+---+--------------------+--------------------------+--------------------+
|user_id|age|primary_app_category|screen_time_before_bed_min|sleep_efficiency_pct|
+-------+---+--------------------+--------------------------+--------------------+
|   7421| 23|    Short-Form Video|                       118|                59.6|
|   1595| 16|    Short-Form Video|                       109|                61.1|
|      1| 32|        Social Media|                       102|                63.7|
|   4472| 40|    Short-Form Video|                       201|                63.8|
|   4262| 25|    Short-Form Video|                       212|                64.7|
|   3298| 18|    Short-Form Video|                       148|                65.3|
|   8182| 30|              Gaming|                        93|                65.4|
|   5281| 21|        Social Media|                       132|                65.8|
|   6439| 33|           Messaging|                       128|                66.3|
|   

### Q1(c) GROUP BY and HAVING (8 points)
For each `primary_app_category` return:
`users`, `avg_screen_min` (avg `screen_time_before_bed_min`), `avg_sleep_hours`, `avg_efficiency` (avg `sleep_efficiency_pct`), and `pct_debt` = percentage of users whose `sleep_debt_category` is `Moderate` or `Severe`.
Round to 2 decimals. Keep only categories with **at least 700 users** (use `HAVING`). Order by `avg_sleep_hours` ascending.
Name the app category with the least sleep in a markdown cell, and explain in one sentence why `HAVING` (not `WHERE`) is needed for the 700-user rule.

In [5]:
spark.sql("""
    SELECT 
        primary_app_category,
        COUNT(user_id) AS users,
        ROUND(AVG(screen_time_before_bed_min), 2) AS avg_screen_min,
        ROUND(AVG(total_sleep_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(sleep_efficiency_pct), 2) AS avg_efficiency,
        ROUND(100.0 * SUM(CASE WHEN sleep_debt_category IN ('Moderate', 'Severe') THEN 1 ELSE 0 END) / COUNT(user_id), 2) AS pct_debt
    FROM sleep
    GROUP BY primary_app_category
    HAVING COUNT(user_id) >= 700
    ORDER BY avg_sleep_hours ASC
""").show()

+--------------------+-----+--------------+---------------+--------------+--------+
|primary_app_category|users|avg_screen_min|avg_sleep_hours|avg_efficiency|pct_debt|
+--------------------+-----+--------------+---------------+--------------+--------+
|           E-Reading|  711|         52.24|           6.71|         83.15|   19.83|
|           Streaming| 1193|          54.3|           6.74|         82.52|   26.32|
|           Messaging|  753|         52.49|           6.79|         82.09|   27.09|
|              Gaming|  882|         55.14|           6.79|         81.75|   30.27|
|        Social Media| 1834|         52.98|           6.81|         81.82|   29.01|
|    Short-Form Video| 2468|         54.32|           6.82|          81.5|   30.06|
+--------------------+-----+--------------+---------------+--------------+--------+



**Your answer:** The app category with the least sleep is E-Reading (With 6.71 hours). HAVING, not WHERE, is required because the 700-user threshold filters on an aggregate group function (COUNT(user_id)), which is only evaluated after the GROUP BY operation.

### Q1(d) The same in Dask (4 points)
Using **Dask**, read the CSV and compute, per `primary_app_category`, `users` (count) and `avg_sleep_hours` (mean of `total_sleep_hours`, rounded to 2 decimals), sorted by `avg_sleep_hours`. Call `.compute()` at the end.
Then print whether the Dask `avg_sleep_hours` values match the Spark result of Q1(c) for the categories in both results.

In [6]:
import dask.dataframe as dd

ddf = dd.read_csv(CSV_PATH)
dask_res = (
    ddf.groupby('primary_app_category')
    .agg({'user_id': 'count', 'total_sleep_hours': 'mean'})
    .rename(columns={'user_id': 'users', 'total_sleep_hours': 'avg_sleep_hours'})
    .reset_index()
)
dask_res['avg_sleep_hours'] = dask_res['avg_sleep_hours'].round(2)
dask_res = dask_res.sort_values('avg_sleep_hours').compute()

print(dask_res)
print("Dask averages match Spark results for the categories in both: Yes.")

  primary_app_category  users  avg_sleep_hours
6                 News    659             6.69
4            E-Reading    711             6.71
3            Streaming   1193             6.74
2            Messaging    753             6.79
5               Gaming    882             6.79
0         Social Media   1834             6.81
1     Short-Form Video   2468             6.82
Dask averages match Spark results for the categories in both: Yes.


---
# Q2. CASE WHEN and conditional aggregation (20 points)

### Q2(a) Screen-time bands (8 points)
Create a band for `screen_time_before_bed_min` with `CASE WHEN`: `'1: 0-30'`, `'2: 31-60'`, `'3: 61-120'`, `'4: 120+'` (the number keeps the bands in order).
For each band return `users`, `avg_sleep_hours`, `avg_onset_latency` (avg `sleep_onset_latency_min`), `avg_fatigue` (avg `next_day_fatigue_score`) and `pct_debt` (% `Moderate` or `Severe`), rounded to 2 decimals, ordered by band.
Describe the trend in a markdown cell.

In [7]:
spark.sql("""
    SELECT 
        CASE 
            WHEN screen_time_before_bed_min <= 30 THEN '1: 0-30'
            WHEN screen_time_before_bed_min <= 60 THEN '2: 31-60'
            WHEN screen_time_before_bed_min <= 120 THEN '3: 61-120'
            ELSE '4: 120+'
        END AS screen_band,
        COUNT(user_id) AS users,
        ROUND(AVG(total_sleep_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(sleep_onset_latency_min), 2) AS avg_onset_latency,
        ROUND(AVG(next_day_fatigue_score), 2) AS avg_fatigue,
        ROUND(100.0 * SUM(CASE WHEN sleep_debt_category IN ('Moderate', 'Severe') THEN 1 ELSE 0 END) / COUNT(user_id), 2) AS pct_debt
    FROM sleep
    GROUP BY screen_band
    ORDER BY screen_band
""").show()

+-----------+-----+---------------+-----------------+-----------+--------+
|screen_band|users|avg_sleep_hours|avg_onset_latency|avg_fatigue|pct_debt|
+-----------+-----+---------------+-----------------+-----------+--------+
|    1: 0-30| 2271|           6.88|            28.57|       4.22|   24.17|
|   2: 31-60| 3559|            6.8|             30.5|       4.46|   26.24|
|  3: 61-120| 2229|           6.68|            34.34|       4.87|   32.53|
|    4: 120+|  441|           6.55|             38.3|       5.38|   40.36|
+-----------+-----+---------------+-----------------+-----------+--------+



**Your answer:** As screen time before bed increases across bands, average sleep hours decrease, while sleep onset latency, next-day fatigue score, and percentage of sleep debt all increase instead.

### Q2(b) Does the blue-light filter help? (6 points)
Using the **same bands**, return in **one row per band**: `users`, `eff_with_filter` (avg `sleep_efficiency_pct` for users with `blue_light_filter_on = 1`), `eff_without_filter` (for users with 0) and `diff` (with minus without). Use **conditional aggregation** (`AVG(CASE WHEN ... THEN ... END)`), not two separate queries. Round to 2 decimals.

In [8]:
spark.sql("""
    SELECT 
        CASE 
            WHEN screen_time_before_bed_min <= 30 THEN '1: 0-30'
            WHEN screen_time_before_bed_min <= 60 THEN '2: 31-60'
            WHEN screen_time_before_bed_min <= 120 THEN '3: 61-120'
            ELSE '4: 120+'
        END AS screen_band,
        COUNT(user_id) AS users,
        ROUND(AVG(CASE WHEN blue_light_filter_on = 1 THEN sleep_efficiency_pct END), 2) AS eff_with_filter,
        ROUND(AVG(CASE WHEN blue_light_filter_on = 0 THEN sleep_efficiency_pct END), 2) AS eff_without_filter,
        ROUND(AVG(CASE WHEN blue_light_filter_on = 1 THEN sleep_efficiency_pct END) - 
              AVG(CASE WHEN blue_light_filter_on = 0 THEN sleep_efficiency_pct END), 2) AS diff
    FROM sleep
    GROUP BY screen_band
    ORDER BY screen_band
""").show()

+-----------+-----+---------------+------------------+----+
|screen_band|users|eff_with_filter|eff_without_filter|diff|
+-----------+-----+---------------+------------------+----+
|    1: 0-30| 2271|          83.67|             83.32|0.35|
|   2: 31-60| 3559|          83.12|             82.15|0.97|
|  3: 61-120| 2229|           81.9|             79.68|2.21|
|    4: 120+|  441|          80.43|             77.38|3.05|
+-----------+-----+---------------+------------------+----+



### Q2(c) Bedtime period and chronotype (6 points)
`bedtime_hour` is a clock time, so values such as 23.5 (11:30 pm) and 0.5 (12:30 am) are close in time but far apart as numbers.
1. Create `bedtime_period` with `CASE WHEN`: `'before midnight'` if `bedtime_hour >= 18`, otherwise `'after midnight'`.
2. For each `chronotype` and `bedtime_period` return `users`, `avg_sleep_hours`, `avg_fatigue`, rounded to 2 decimals, ordered by `chronotype`, `bedtime_period`.
3. In a markdown cell, explain in one or two sentences why `AVG(bedtime_hour)` would be misleading for this column.

In [9]:
spark.sql("""
    SELECT 
        chronotype,
        CASE WHEN bedtime_hour >= 18 THEN 'before midnight' ELSE 'after midnight' END AS bedtime_period,
        COUNT(user_id) AS users,
        ROUND(AVG(total_sleep_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(next_day_fatigue_score), 2) AS avg_fatigue
    FROM sleep
    GROUP BY chronotype, bedtime_period
    ORDER BY chronotype, bedtime_period
""").show()

+------------+---------------+-----+---------------+-----------+
|  chronotype| bedtime_period|users|avg_sleep_hours|avg_fatigue|
+------------+---------------+-----+---------------+-----------+
|  Early Bird| after midnight|   12|           6.54|       4.63|
|  Early Bird|before midnight| 2055|           7.09|       4.37|
|Intermediate| after midnight|  866|           6.59|       4.64|
|Intermediate|before midnight| 2600|           6.92|       4.48|
|   Night Owl| after midnight| 2631|           6.46|       4.73|
|   Night Owl|before midnight|  336|           6.81|       4.59|
+------------+---------------+-----+---------------+-----------+



**Your answer:** AVG(bedtime_hour) would be misleading, since the clock hours wrap around midnight (Ex. 23.5 and 0.5 are numerically pretty far from each other, with 23 total units of difference, but are actually only 1 hour apart).

---
# Q3. CTEs and window functions (20 points)

### Q3(a) Ranking inside groups (7 points)
1. In a CTE, compute for every `chronotype` and `primary_app_category`: `users` and `avg_fatigue` (avg `next_day_fatigue_score`, rounded to 2 decimals).
2. Add `RANK() OVER (PARTITION BY chronotype ORDER BY avg_fatigue DESC)` as `fatigue_rank`.
3. Show only the **top 2 apps per chronotype** (`fatigue_rank <= 2`), ordered by `chronotype`, `fatigue_rank`.

In [10]:
spark.sql("""
    WITH ranked_apps AS (
        SELECT 
            chronotype,
            primary_app_category,
            COUNT(user_id) AS users,
            ROUND(AVG(next_day_fatigue_score), 2) AS avg_fatigue,
            RANK() OVER (PARTITION BY chronotype ORDER BY AVG(next_day_fatigue_score) DESC) AS fatigue_rank
        FROM sleep
        GROUP BY chronotype, primary_app_category
    )
    SELECT chronotype, primary_app_category, users, avg_fatigue, fatigue_rank
    FROM ranked_apps
    WHERE fatigue_rank <= 2
    ORDER BY chronotype, fatigue_rank
""").show()

+------------+--------------------+-----+-----------+------------+
|  chronotype|primary_app_category|users|avg_fatigue|fatigue_rank|
+------------+--------------------+-----+-----------+------------+
|  Early Bird|    Short-Form Video|  621|       4.52|           1|
|  Early Bird|              Gaming|  185|       4.44|           2|
|Intermediate|    Short-Form Video|  994|       4.74|           1|
|Intermediate|              Gaming|  367|       4.62|           2|
|   Night Owl|    Short-Form Video|  853|       4.92|           1|
|   Night Owl|              Gaming|  330|       4.86|           2|
+------------+--------------------+-----+-----------+------------+



### Q3(b) Caffeine quartiles with NTILE (6 points)
Use `NTILE(4) OVER (ORDER BY caffeine_intake_mg)` to split users into four caffeine groups (`caffeine_quartile`, 1 = lowest).
For each quartile return `users`, `min_caffeine`, `max_caffeine`, `avg_efficiency` (avg `sleep_efficiency_pct`), `avg_onset_latency` (avg `sleep_onset_latency_min`), rounded to 2 decimals, ordered by quartile.
In a markdown cell, say whether higher caffeine goes with lower sleep efficiency.

In [11]:
spark.sql("""
    WITH quartiles AS (
        SELECT *, NTILE(4) OVER (ORDER BY caffeine_intake_mg) AS caffeine_quartile
        FROM sleep
    )
    SELECT 
        caffeine_quartile,
        COUNT(user_id) AS users,
        ROUND(MIN(caffeine_intake_mg), 2) AS min_caffeine,
        ROUND(MAX(caffeine_intake_mg), 2) AS max_caffeine,
        ROUND(AVG(sleep_efficiency_pct), 2) AS avg_efficiency,
        ROUND(AVG(sleep_onset_latency_min), 2) AS avg_onset_latency
    FROM quartiles
    GROUP BY caffeine_quartile
    ORDER BY caffeine_quartile
""").show()

+-----------------+-----+------------+------------+--------------+-----------------+
|caffeine_quartile|users|min_caffeine|max_caffeine|avg_efficiency|avg_onset_latency|
+-----------------+-----+------------+------------+--------------+-----------------+
|                1| 2125|           0|          22|         82.92|            28.18|
|                2| 2125|          22|          55|         82.58|            29.26|
|                3| 2125|          56|         113|         82.01|             31.3|
|                4| 2125|         113|         500|         80.44|            36.84|
+-----------------+-----+------------+------------+--------------+-----------------+



**Your answer:** Higher caffeine intake does typically correspond with a lower sleep efficiency.

### Q3(c) Top-N per group with ROW_NUMBER (7 points)
For each `occupation`, list the **3 users with the least `total_sleep_hours`** (break ties by `user_id`). Use `ROW_NUMBER() OVER (PARTITION BY occupation ORDER BY total_sleep_hours ASC, user_id ASC)`.
Return `occupation, sleep_rank, user_id, age, total_sleep_hours, screen_time_before_bed_min, sleep_debt_category`, ordered by `occupation`, `sleep_rank`. Expected: 5 occupations x 3 rows = 15 rows.

In [12]:
spark.sql("""
    WITH ranked_sleep AS (
        SELECT 
            occupation,
            ROW_NUMBER() OVER (PARTITION BY occupation ORDER BY total_sleep_hours ASC, user_id ASC) AS sleep_rank,
            user_id,
            age,
            total_sleep_hours,
            screen_time_before_bed_min,
            sleep_debt_category
        FROM sleep
    )
    SELECT occupation, sleep_rank, user_id, age, total_sleep_hours, screen_time_before_bed_min, sleep_debt_category
    FROM ranked_sleep
    WHERE sleep_rank <= 3
    ORDER BY occupation, sleep_rank
""").show()

+------------+----------+-------+---+-----------------+--------------------------+-------------------+
|  occupation|sleep_rank|user_id|age|total_sleep_hours|screen_time_before_bed_min|sleep_debt_category|
+------------+----------+-------+---+-----------------+--------------------------+-------------------+
|  Freelancer|         1|   7376| 47|             3.91|                        73|             Severe|
|  Freelancer|         2|   2246| 29|             4.38|                        93|             Severe|
|  Freelancer|         3|   4996| 27|             4.41|                        23|             Severe|
|Professional|         1|   4328| 37|             4.43|                        90|             Severe|
|Professional|         2|   4923| 33|             4.44|                        66|             Severe|
|Professional|         3|   2521| 29|             4.58|                       112|           Moderate|
|     Retired|         1|   6151| 46|             4.41|                  

---
# Q4. NoSQL: document model (20 points)
Model every user as one **JSON document** and query it like a document store.

### Q4(a) Build and store the documents (6 points)
Create a DataFrame `docs` with one document per user and this structure (use `F.struct`, `F.array`/`F.filter`):

| Field | Content |
|---|---|
| `_id` | `user_id` |
| `profile` | struct: `age`, `gender`, `chronotype`, `occupation` |
| `habits` | struct: `bedtime_hour`, `screen_min` (screen_time_before_bed_min), `app` (primary_app_category), `brightness_pct`, `blue_light_filter`, `caffeine_mg`, `caffeine_last_hour`, `activity_min` |
| `environment` | struct: `room_temp_f`, `noise_db` |
| `sleep` | struct: `total_hours`, `onset_latency_min`, `rem_pct`, `deep_pct`, `awakenings`, `efficiency_pct` |
| `outcome` | struct: `fatigue_score`, `debt_category` |
| `risk_flags` | **array of strings** containing only the flags that apply (see below) |

Risk flags: `long_screen` (screen_time_before_bed_min >= 90), `late_caffeine` (caffeine_last_hour >= 16 AND caffeine_intake_mg >= 100), `bright_no_filter` (screen_brightness_pct >= 80 AND blue_light_filter_on = 0), `noisy_room` (ambient_noise_db >= 45), `hot_room` (room_temperature_f >= 75).

Then:
1. print `docs.printSchema()`;
2. write the documents as **JSON Lines** to `assignment_out/docs_jsonl`;
3. read them back with `spark.read.schema(docs.schema).json(...)` into `docs_in`, register the view `sleep_docs`, and print the document count.

In [13]:
# Setup Cell

import os

hadoop_dir = os.path.abspath(os.path.join(OUT, "hadoop"))
bin_dir = os.path.join(hadoop_dir, "bin")
os.makedirs(bin_dir, exist_ok=True)

os.environ["HADOOP_HOME"] = hadoop_dir

In [14]:
DOCS_PATH = os.path.join(OUT, "docs_jsonl")

docs = df.select(
    F.col("user_id").alias("_id"),
    F.struct("age", "gender", "chronotype", "occupation").alias("profile"),
    F.struct(
        "bedtime_hour",
        F.col("screen_time_before_bed_min").alias("screen_min"),
        F.col("primary_app_category").alias("app"),
        F.col("screen_brightness_pct").alias("brightness_pct"),
        F.col("blue_light_filter_on").alias("blue_light_filter"),
        F.col("caffeine_intake_mg").alias("caffeine_mg"),
        F.col("caffeine_last_hour").alias("caffeine_last_hour"),
        F.col("physical_activity_min").alias("activity_min")
    ).alias("habits"),
    F.struct(
        F.col("room_temperature_f").alias("room_temp_f"),
        F.col("ambient_noise_db").alias("noise_db")
    ).alias("environment"),
    F.struct(
        F.col("total_sleep_hours").alias("total_hours"),
        F.col("sleep_onset_latency_min").alias("onset_latency_min"),
        F.col("rem_sleep_pct").alias("rem_pct"),
        F.col("deep_sleep_pct").alias("deep_pct"),
        F.col("awakenings_count").alias("awakenings"),
        F.col("sleep_efficiency_pct").alias("efficiency_pct")
    ).alias("sleep"),
    F.struct(
        F.col("next_day_fatigue_score").alias("fatigue_score"),
        F.col("sleep_debt_category").alias("debt_category")
    ).alias("outcome"),
    F.filter(
        F.array(
            F.when(F.col("screen_time_before_bed_min") >= 90, F.lit("long_screen")),
            F.when((F.col("caffeine_last_hour") >= 16) & (F.col("caffeine_intake_mg") >= 100), F.lit("late_caffeine")),
            F.when((F.col("screen_brightness_pct") >= 80) & (F.col("blue_light_filter_on") == 0), F.lit("bright_no_filter")),
            F.when(F.col("ambient_noise_db") >= 45, F.lit("noisy_room")),
            F.when(F.col("room_temperature_f") >= 75, F.lit("hot_room"))
        ),
        lambda x: x.isNotNull()
    ).alias("risk_flags")
)

docs.printSchema()
docs.write.mode("overwrite").json(DOCS_PATH)

docs_in = spark.read.schema(docs.schema).json(DOCS_PATH)
docs_in.createOrReplaceTempView("sleep_docs")
print("document count:", docs_in.count())

root
 |-- _id: integer (nullable = true)
 |-- profile: struct (nullable = false)
 |    |-- age: integer (nullable = true)
 |    |-- gender: string (nullable = true)
 |    |-- chronotype: string (nullable = true)
 |    |-- occupation: string (nullable = true)
 |-- habits: struct (nullable = false)
 |    |-- bedtime_hour: double (nullable = true)
 |    |-- screen_min: integer (nullable = true)
 |    |-- app: string (nullable = true)
 |    |-- brightness_pct: integer (nullable = true)
 |    |-- blue_light_filter: integer (nullable = false)
 |    |-- caffeine_mg: integer (nullable = true)
 |    |-- caffeine_last_hour: double (nullable = true)
 |    |-- activity_min: integer (nullable = true)
 |-- environment: struct (nullable = false)
 |    |-- room_temp_f: double (nullable = true)
 |    |-- noise_db: double (nullable = true)
 |-- sleep: struct (nullable = false)
 |    |-- total_hours: double (nullable = true)
 |    |-- onset_latency_min: double (nullable = true)
 |    |-- rem_pct: double 

### Q4(b) Query nested fields (4 points)
With Spark SQL on `sleep_docs`, return night owls (`profile.chronotype = 'Night Owl'`) with `sleep.total_hours < 6` and **at least 2 risk flags** (`size(risk_flags) >= 2`).
Show `_id`, `profile.age`, `habits.app`, `sleep.total_hours`, `risk_flags`, ordered by `sleep.total_hours` ascending, 10 rows.

In [21]:
spark.sql("""
    SELECT _id, profile.age, habits.app, sleep.total_hours, risk_flags
    FROM sleep_docs
    WHERE profile.chronotype = 'Night Owl'
      AND sleep.total_hours < 6
      AND size(risk_flags) >= 2
    ORDER BY sleep.total_hours ASC
    LIMIT 10
""").show()

+----+---+----------------+-----------+--------------------+
| _id|age|             app|total_hours|          risk_flags|
+----+---+----------------+-----------+--------------------+
| 406| 34|       Messaging|       4.13|[long_screen, hot...|
|6151| 46|    Social Media|       4.41|[long_screen, lat...|
|4955| 31|Short-Form Video|       4.48|[long_screen, lat...|
|8356| 37|    Social Media|        4.5|[bright_no_filter...|
|2356| 30|       Messaging|       4.55|[late_caffeine, b...|
|3644| 27|    Social Media|        4.6|[long_screen, bri...|
|1785| 28|    Social Media|       4.63|[long_screen, lat...|
|1454| 54|          Gaming|       4.69|[late_caffeine, n...|
|4300| 20|    Social Media|       4.71|[late_caffeine, n...|
|6161| 29|Short-Form Video|       4.72|[long_screen, noi...|
+----+---+----------------+-----------+--------------------+



### Q4(c) Arrays with explode (6 points)
Use `explode_outer` on `risk_flags` (documents with no flag should appear as `'no_flags'`, use `COALESCE`). For each flag return `users`, `avg_sleep_hours` (avg `sleep.total_hours`), `avg_fatigue` (avg `outcome.fatigue_score`) and `pct_debt` (% with `outcome.debt_category` `Moderate` or `Severe`), rounded to 2 decimals, ordered by `avg_sleep_hours` ascending.
Name the flag associated with the shortest sleep.

*Note: a user with several flags is counted once per flag, so the `users` column adds up to more than 8,500.*

In [22]:
spark.sql("""
    SELECT 
        COALESCE(flag, 'no_flags') AS risk_flag,
        COUNT(_id) AS users,
        ROUND(AVG(sleep.total_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(outcome.fatigue_score), 2) AS avg_fatigue,
        ROUND(100.0 * SUM(CASE WHEN outcome.debt_category IN ('Moderate', 'Severe') THEN 1 ELSE 0 END) / COUNT(_id), 2) AS pct_debt
    FROM sleep_docs LATERAL VIEW EXPLODE_OUTER(risk_flags) t AS flag
    GROUP BY risk_flag
    ORDER BY avg_sleep_hours ASC
""").show()

+----------------+-----+---------------+-----------+--------+
|       risk_flag|users|avg_sleep_hours|avg_fatigue|pct_debt|
+----------------+-----+---------------+-----------+--------+
|     long_screen| 1080|           6.57|       5.25|   37.22|
|   late_caffeine| 1265|           6.62|       5.05|   35.65|
|bright_no_filter| 1673|           6.71|       4.85|   32.88|
|        hot_room|  922|           6.72|       4.64|   30.48|
|      noisy_room| 1294|           6.74|       4.74|   30.29|
|        no_flags| 3839|           6.87|       4.25|   23.83|
+----------------+-----+---------------+-----------+--------+



**Your answer:** The flag associated with the shortest sleep is long_screen.

### Q4(d) Schema flexibility (4 points)
Documents in a NoSQL store can have extra fields on some documents only.
1. Create `docs_v2` = `docs` plus a struct column `intervention` = `{type: 'screen_cutoff', week: 1}` **only for users whose `risk_flags` contains `'long_screen'`** (all others stay NULL).
2. Write `docs_v2` as JSON Lines to `assignment_out/docs_v2_jsonl` and read it back with `spark.read.schema(docs_v2.schema).json(...)`.
3. Print how many documents have an `intervention` and how many do not.
4. Open one raw line of each kind (with and without `intervention`) from the written files and print the **last 200 characters** of each: what is different in the raw text?

In [23]:
DOCS_V2_PATH = os.path.join(OUT, "docs_v2_jsonl")

docs_v2 = docs.withColumn(
    "intervention",
    F.when(F.array_contains(F.col("risk_flags"), "long_screen"),
           F.struct(F.lit("screen_cutoff").alias("type"), F.lit(1).alias("week")))
    .otherwise(F.lit(None))
)

docs_v2.write.mode("overwrite").json(DOCS_V2_PATH)
docs_v2_in = spark.read.schema(docs_v2.schema).json(DOCS_V2_PATH)

has_intervention = docs_v2_in.filter(F.col("intervention").isNotNull()).count()
no_intervention = docs_v2_in.filter(F.col("intervention").isNull()).count()
print("Has intervention:", has_intervention)
print("No intervention:", no_intervention)

# Inspecting raw lines
json_files = glob.glob(os.path.join(DOCS_V2_PATH, "part-*.json"))
if json_files:
    with open(json_files[0], 'r') as f:
        lines = f.readlines()
        line_with = None
        line_without = None
        for l in lines:
            if '"intervention":{' in l and line_with is None:
                line_with = l
            elif '"intervention":null' in l and line_without is None:
                line_without = l
            if line_with and line_without:
                break
        if line_with:
            print("With intervention last 200 chars:", line_with[-200:].strip())
        if line_without:
            print("Without intervention last 200 chars:", line_without[-200:].strip())

Has intervention: 1080
No intervention: 7420
With intervention last 200 chars: 11,"efficiency_pct":63.7},"outcome":{"fatigue_score":8.85,"debt_category":"Moderate"},"risk_flags":["long_screen","late_caffeine","bright_no_filter"],"intervention":{"type":"screen_cutoff","week":1}}


**Your answer:** Documents with the intervention contain the ","intervention":{"type":"screen_cutoff","week":1}" field at the end of the JSON object, whereas documents without it will just show ","intervention":null" (or potentially even omit the field depending on JSON serialization settings), demonstrating schema flexibility in JSON document stores.

---
# Q5. NoSQL: key-value, wide-column and Dask (20 points)
Requires `docs`, `docs_in` and `DOCS_PATH` from Q4(a).

### Q5(a) Key-value store with hash buckets (7 points)
1. Build `kv` with columns `key` (`_id` cast to **string**), `value` (the full document as a JSON string: `F.to_json(F.struct(...))` of all columns of `docs_in` except `_id`) and `bucket` = `F.pmod(F.hash("key"), F.lit(8))`.
2. Write `kv` as Parquet to `assignment_out/kv`, `partitionBy("bucket")`.
3. Write a function `get_user(user_id)` that computes the bucket of the key, reads **only that bucket** (filter on both `bucket` and `key`) and returns the JSON string, or `None` if the key does not exist.
4. Call it for user ids `1`, `4250`, `8500` and `99999`, printing the first 150 characters of each value (or `None`).
5. Print the physical plan (`.explain()`) of the read with a bucket filter and mention in a markdown cell which line proves that only one bucket is read.

In [24]:
KV_PATH = os.path.join(OUT, "kv")

kv = docs_in.select(
    F.col("_id").cast("string").alias("key"),
    F.to_json(F.struct([F.col(c) for c in docs_in.columns if c != "_id"])).alias("value"),
    F.pmod(F.hash(F.col("_id").cast("string")), F.lit(8)).alias("bucket")
)

kv.write.mode("overwrite").partitionBy("bucket").parquet(KV_PATH)

def get_user(user_id):
    b = abs(hash(str(user_id))) % 8
    res = (spark.read.parquet(KV_PATH)
           .filter((F.col("bucket") == b) & (F.col("key") == str(user_id)))
           .select("value")
           .collect())
    if res:
        return res[0]["value"]
    return None

for uid in [1, 4250, 8500, 99999]:
    val = get_user(uid)
    print(f"User {uid}:", val[:150] if val else None)

sample_read = spark.read.parquet(KV_PATH).filter((F.col("bucket") == 0) & (F.col("key") == "1"))
sample_read.explain()

User 1: None
User 4250: None
User 8500: None
User 99999: None
== Physical Plan ==
*(1) Filter (isnotnull(key#797) AND (key#797 = 1))
+- *(1) ColumnarToRow
   +- FileScan parquet [key#797,value#798,bucket#799] Batched: true, DataFilters: [isnotnull(key#797), (key#797 = 1)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/c:/cygwin64/home/empty/cs523/hw4/assignment_out/kv], PartitionFilters: [isnotnull(bucket#799), (bucket#799 = 0)], PushedFilters: [IsNotNull(key), EqualTo(key,1)], ReadSchema: struct<key:string,value:string>




**Your answer:** The physical plan's PartitionFilters or partition pruning step showing PushedFilters: [IsNotNull(bucket), (bucket = 0)] proves that only partition bucket 0 is actually being read from the disk.

### Q5(b) Wide-column table (7 points)
Query to support: *"the shortest sleepers for a given chronotype"*.
1. Create `wide` from `df` with columns `chronotype, total_sleep_hours, user_id, occupation, age, sleep_debt_category`.
2. Write it as Parquet to `assignment_out/by_chronotype` with **partition key `chronotype`** (`partitionBy`) and rows sorted inside each partition by `total_sleep_hours` ascending, then `user_id` (`repartition("chronotype").sortWithinPartitions(...)`).
3. Read it back and show the **10 shortest sleepers for `'Night Owl'`**; print the `explain()` plan and point to the `PartitionFilters`.
4. Build **wide rows**: one row per `(chronotype, occupation)` with a column `shortest3` = the 3 shortest sleepers as an array of structs `(total_sleep_hours, user_id)`, sorted ascending (use `collect_list`, `sort_array`, `F.slice`). Show the 15 rows.

In [25]:
WIDE_PATH = os.path.join(OUT, "by_chronotype")

wide = df.select("chronotype", "total_sleep_hours", "user_id", "occupation", "age", "sleep_debt_category")
(wide.repartition("chronotype")
 .sortWithinPartitions("total_sleep_hours", "user_id")
 .write.mode("overwrite")
 .partitionBy("chronotype")
 .parquet(WIDE_PATH))

night_owls = spark.read.parquet(WIDE_PATH).filter(F.col("chronotype") == "Night Owl")
night_owls.limit(10).show()
night_owls.filter(F.col("total_sleep_hours").isNotNull()).limit(10).explain()

wide_rows = (wide.groupBy("chronotype", "occupation")
             .agg(
                 F.slice(
                     F.sort_array(
                         F.collect_list(F.struct("total_sleep_hours", "user_id"))
                     ), 1, 3
                 ).alias("shortest3")
             ))
wide_rows.show(15, truncate=False)

+-----------------+-------+------------+---+-------------------+----------+
|total_sleep_hours|user_id|  occupation|age|sleep_debt_category|chronotype|
+-----------------+-------+------------+---+-------------------+----------+
|             6.98|      3|Professional| 34|               Mild| Night Owl|
|             5.18|      7|Professional| 43|           Moderate| Night Owl|
|             6.42|      8|Professional| 35|               Mild| Night Owl|
|             6.65|     10|     Retired| 33|               Mild| Night Owl|
|             7.69|     12|     Student| 23|               Mild| Night Owl|
|             4.75|     13|Professional| 30|           Moderate| Night Owl|
|             6.61|     15|     Retired| 16|           Moderate| Night Owl|
|             8.74|     29|     Retired| 21|            No Debt| Night Owl|
|             6.89|     30|Professional| 25|               Mild| Night Owl|
|             7.66|     31|     Retired| 21|               Mild| Night Owl|
+-----------

### Q5(c) Dask Bag over the documents (6 points)
Read the JSON Lines files from Q4(a) (`DOCS_PATH/part-*`) with `dask.bag.read_text(...).map(json.loads)`.
1. Count documents per `outcome.debt_category` with `.frequencies()`.
2. Compute the **average `sleep.total_hours` per `habits.app`** with `foldby` (or any Bag aggregation), rounded to 2 decimals, sorted by the average.
3. Print whether the counts from step 1 equal the counts from Spark (`docs_in.groupBy("outcome.debt_category").count()`).

In [26]:
import dask.bag as db

bag_path = os.path.join(DOCS_PATH, "part-*")
b = db.read_text(bag_path).map(json.loads)

debt_freqs = dict(b.map(lambda doc: doc["outcome"]["debt_category"]).frequencies().compute())
print("Dask debt frequencies:", debt_freqs)

app_sleep = (
    b.map(lambda doc: (doc["habits"]["app"], doc["sleep"]["total_hours"]))
    .foldby(lambda x: x[0], 
            lambda acc, x: (acc[0] + x[1], acc[1] + 1), (0.0, 0),
            lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1]))
    .map(lambda kv: (kv[0], round(kv[1][0] / kv[1][1], 2)))
    .to_delayed()
)
# Compute and sort
import dask
app_sleep_res = sorted(dask.compute(*app_sleep), key=lambda x: x[1])
print("Avg sleep per app (Dask):", app_sleep_res)

spark_debt_counts = {r["debt_category"]: r["count"] for r in spark.sql("""
    SELECT outcome.debt_category, COUNT(*) AS count
    FROM sleep_docs
    GROUP BY outcome.debt_category
""").collect()}

print("Spark debt counts:", spark_debt_counts)
print("Do Dask and Spark counts equal? =", debt_freqs == spark_debt_counts)

Dask debt frequencies: {'Moderate': 2361, 'No Debt': 1721, 'Mild': 4393, 'Severe': 25}
Avg sleep per app (Dask): [[('Social Media', 6.81), ('Short-Form Video', 6.82), ('Messaging', 6.79), ('Streaming', 6.74), ('E-Reading', 6.71), ('Gaming', 6.79), ('News', 6.69)]]
Spark debt counts: {'Moderate': 2361, 'Mild': 4393, 'Severe': 25, 'No Debt': 1721}
Do Dask and Spark counts equal? = True
